In [1]:
import pandas as pd
from scipy.stats.mstats import trimmed_var
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline

In [2]:
def wrangle(filepath):
  df = pd.read_csv(filepath)
  mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
  df = df[mask]

  return df


In [4]:
df = wrangle("/content/drive/MyDrive/SCFP2019.csv")

print(f"DATAFRAME SHAPE: {df.shape}")

df.head()

DATAFRAME SHAPE: (4418, 351)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,LF,LIFECL,FAMSTRUCT,RACECL,RACECL4,RACE,OCCAT1,OCCAT2,INDCAT,FOODHOME,FOODAWAY,FOODDELV,RENT,INCOME,WAGEINC,BUSSEFARMINC,INTDIVINC,KGINC,SSRETINC,TRANSFOTHINC,PENACCTWD,NORMINC,WSAVED,SAVED,SAVRES1,SAVRES2,SAVRES3,SAVRES4,SAVRES5,SAVRES6,...,PAYPEN6,TPAY,MORTPAY,CONSPAY,REVPAY,PIRTOTAL,PIRMORT,PIRCONS,PIRREV,PIR40,PLOAN1,PLOAN2,PLOAN3,PLOAN4,PLOAN5,PLOAN6,PLOAN7,PLOAN8,LLOAN1,LLOAN2,LLOAN3,LLOAN4,LLOAN5,LLOAN6,LLOAN7,LLOAN8,LLOAN9,LLOAN10,LLOAN11,LLOAN12,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
5,2,21,3790.476607,1,50,3,8,2,1,3,1,3,4,1,1,1,4,4,4,5760,720,0,1100.0,38688.480260,11199.296917,0.0,0.0,0.0,9264.872904,16798.945376,0.0,38688.480260,1,0,0,0,0,0,0,0,...,0,400.0,0.0,400.0,0.0,0.124068,0.0,0.124068,0.0,0,0.0,0.0,12200,0,0.0,0,0,0,0.0,0,12200,0,0,0,0,0,0,0,0,0,1,2,1,2,1,1,4,4,2,2
6,2,22,3798.868505,1,50,3,8,2,1,3,1,3,4,1,1,1,4,4,4,5760,720,0,1100.0,37670.362358,11199.296917,0.0,0.0,0.0,9366.684694,16595.321796,0.0,37670.362358,1,0,0,0,0,0,0,0,...,0,390.0,0.0,390.0,0.0,0.124236,0.0,0.124236,0.0,0,0.0,0.0,12600,0,0.0,0,0,0,0.0,0,12600,0,0,0,0,0,0,0,0,0,1,2,1,2,1,1,4,3,2,2
7,2,23,3799.468393,1,50,3,8,2,1,3,1,3,4,1,1,1,4,4,4,5760,720,0,1100.0,38688.480260,11199.296917,0.0,0.0,0.0,9264.872904,17409.816117,0.0,38688.480260,1,0,0,0,0,0,0,0,...,0,390.0,0.0,390.0,0.0,0.120966,0.0,0.120966,0.0,0,0.0,0.0,15300,0,0.0,0,0,0,0.0,0,15300,0,0,0,0,0,0,0,0,0,1,2,1,2,1,1,4,4,2,2
8,2,24,3788.076005,1,50,3,8,2,1,3,1,3,4,1,1,1,4,4,4,5760,720,0,1000.0,38688.480260,12217.414819,0.0,0.0,0.0,9264.872904,17409.816117,0.0,38688.480260,1,0,0,0,0,0,0,0,...,0,390.0,0.0,390.0,0.0,0.120966,0.0,0.120966,0.0,0,0.0,0.0,14100,0,0.0,0,0,0,0.0,0,14100,0,0,0,0,0,0,0,0,0,1,2,1,2,1,1,4,4,2,2
9,2,25,3793.066589,1,50,3,8,2,1,3,1,3,4,1,1,1,4,4,4,5760,720,0,1100.0,38688.480260,12217.414819,0.0,0.0,0.0,9366.684694,17002.568956,0.0,38688.480260,1,0,0,0,0,0,0,0,...,0,390.0,0.0,390.0,0.0,0.120966,0.0,0.120966,0.0,0,0.0,0.0,15400,0,0.0,0,0,0,0.0,0,15400,0,0,0,0,0,0,0,0,0,1,2,1,2,1,1,4,4,2,2


# Feature Selection Using Variance

Features with almost no variance dont really tell much about household separate data

## Cluster using Top Ten High Variance Columns



In [5]:
top_ten_var_col = df.var().sort_values(ascending = False).head(10)

print(top_ten_var_col)

ASSET       8.303967e+10
NFIN        5.713939e+10
NETWORTH    4.847029e+10
HOUSES      2.388459e+10
NHNFIN      2.254163e+10
DEBT        1.848252e+10
KGTOTAL     1.346475e+10
BUS         1.256643e+10
ACTBUS      1.251892e+10
PLOAN1      1.140894e+10
dtype: float64


In [6]:
fig = px.bar(
    x = top_ten_var_col,
    y = top_ten_var_col.index,
    orientation = "h",
    title = "Top Ten High Variance Columns"
)

fig.update_layout(
    xaxis_title = "Feature",
    yaxis_title = "Variance"
)

fig.show()


In [11]:
fig = px.box(
    data_frame = df,
    x="NHNFIN",
    title = "Distribution of Non-home, Non-Financial Assets"
)

fig.update_layout(
    xaxis_title = "Value In Dollars"
)

fig.show()

The box plot shows extreme right skew: even after filtering to net worth under $2 million, a few households sit far above the rest. Those extremes are exactly what raw variance over-rewards — and the reason the next step switches to trimmed variance for a more robust ranking.

In [14]:
top_ten_trim_var = (
    df.apply(trimmed_var, limits=(0.1, 0.1)).sort_values(ascending=False).head(10)
)

top_ten_trim_var

,0
ASSET,1.175370e+10
NFIN,8.456442e+09
HOUSES,4.978660e+09
NETWORTH,3.099929e+09
DEBT,3.089865e+09
PLOAN1,1.441968e+09
MRTHEL,1.380468e+09
NH_MORT,1.333125e+09
HOMEEQ,7.338377e+08
WAGEINC,5.550737e+08


In [15]:
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    orientation="h",
    title="Top Ten Trimmed Variance Columns",
)

fig.update_layout(
    xaxis_title="Feature",
)

fig.show()

In [16]:
high_var_cols = top_ten_trim_var.head(5).index.to_list()

high_var_cols

['ASSET', 'NFIN', 'HOUSES', 'NETWORTH', 'DEBT']

In [17]:
X = df[high_var_cols]

print(f"Feature Shapes : {X.shape}")

X.head()

Feature Shapes : (4418, 5)


,ASSET,NFIN,HOUSES,NETWORTH,DEBT
5,5490.0,3900.0,0.0,-6710.0,12200.0
6,7890.0,6300.0,0.0,-4710.0,12600.0
7,7185.0,5600.0,0.0,-8115.0,15300.0
8,11590.0,10000.0,0.0,-2510.0,14100.0
9,9685.0,8100.0,0.0,-5715.0,15400.0


# Standardization

In [19]:
ss = StandardScaler()

x_scaled_data = ss.fit_transform(X)

X_scaled = pd.DataFrame(x_scaled_data, columns=X.columns)

X_scaled.head()

X_scaled.head()

,ASSET,NFIN,HOUSES,NETWORTH,DEBT
0,-0.498377,-0.474583,-0.48231,-0.377486,-0.445075
1,-0.490047,-0.464541,-0.48231,-0.368401,-0.442132
2,-0.492494,-0.467470,-0.48231,-0.383868,-0.422270
3,-0.477206,-0.449061,-0.48231,-0.358407,-0.431097
4,-0.483818,-0.457010,-0.48231,-0.372966,-0.421534


In [21]:
# Summary
summary = X_scaled.agg(["mean","std"]).round(2)

summary

,ASSET,NFIN,HOUSES,NETWORTH,DEBT
mean,0.0,0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0


# Inertia & Silhoutte Score

In [25]:
n_clusters = range(2,12)
inertia_errors = []
silhouette_scores = []

for k in n_clusters:
  model = make_pipeline(
      StandardScaler(),
      KMeans(n_clusters=k, random_state=42)
  )

  model.fit(X)

  inertia_errors.append(model[-1].inertia_)
  silhouette_scores.append(silhouette_score(X, model[-1].labels_))

print(f"Inertia errors {inertia_errors[:3]}")
print(f"Silhouette scores {silhouette_scores[:3]}")

Inertia errors [11028.058082607145, 7190.703769068724, 6233.106346879284]
Silhouette scores [np.float64(0.7464502937083215), np.float64(0.7042841259119152), np.float64(0.7046977054481021)]


In [26]:
fig = px.line(
    x=n_clusters,
    y=inertia_errors,
    title="Inertia vs Number of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters",
    yaxis_title="Inertia Error"
)

fig.show()

In [27]:
fig = px.line(
    x=n_clusters,
    y=silhouette_scores,
    title="Silhouette Score vs Number of Clusters"
)

fig.update_layout(
    xaxis_title="Number of Clusters",
    yaxis_title="Silhouette Score"
)
fig.show()

# Final Model

In [28]:
final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=4, random_state=42)
)

final_model.fit(X)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=4, random_state=42))])

In [29]:
labels = final_model.named_steps['kmeans'].labels_

labels[:5]

array([1, 1, 1, 1, 1], dtype=int32)

# Cluster Visualization and Interpetation

In [30]:
xgb = X.groupby(labels).mean()

xgb

,ASSET,NFIN,HOUSES,NETWORTH,DEBT
0,4.365447e+05,3.512653e+05,261927.688504,2.314333e+05,205111.440049
1,4.041617e+04,2.825256e+04,14647.482838,1.176536e+04,28650.810927
2,1.251312e+06,1.118355e+06,788306.451613,5.819507e+05,669360.967742
3,1.698713e+06,1.295769e+06,339117.647059,1.432137e+06,266576.274510


In [32]:
fig = px.bar(
    xgb,
    barmode="group",
    title = "Mean Household Finances by Cluster"
    )

fig.update_layout(
    xaxis_title = "Cluster",
    yaxis_title = "Value in Dollars"
)

fig.show()

# Dimensionality Reduction

In [33]:
pca = PCA(n_components=2, random_state=42)

X_t = pca.fit_transform(X_scaled)

x_pca = pd.DataFrame(X_t, columns=["PC1","PC2"])

print(f"X_pca shape: {x_pca.shape}")

x_pca.head()

X_pca shape: (4418, 2)


,PC1,PC2
0,-1.020669,-0.067119
1,-1.006799,-0.073208
2,-1.008007,-0.049070
3,-0.984522,-0.075588
4,-0.993881,-0.057924


In [35]:
fig = px.scatter(
    data_frame=x_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()